# CSV Data Quality Audit Notebook

This notebook provides a comprehensive suite of tools to analyze the quality of CSV files. It handles various common issues like inconsistent separators, multi-line records, encoding problems, and structural anomalies.

In [97]:
import pandas as pd
import numpy as np
import chardet
import csv
import os
from typing import Dict, List, Any
import re
import json


print("Setup complete.")

Setup complete.


## 1. Physical File Inspection

Before loading into Pandas, we need to understand the file's physical structure: encoding and delimiters.

In [98]:

import datetime

def inspect_file_basics(file_path: str) -> Dict[str, Any]:
    """Detect encoding, separator, and basic physical metadata of a CSV file."""
    stats = os.stat(file_path)
    file_size_mb = round(stats.st_size / (1024 * 1024), 4)
    last_mod = datetime.datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    
    with open(file_path, 'rb') as f:
        raw_data = f.read(50000)
        detection = chardet.detect(raw_data)
        encoding = detection['encoding']
        confidence = detection['confidence']
    
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            content = f.read(50000)
            dialect = csv.Sniffer().sniff(content)
            delimiter = dialect.delimiter
            has_header = csv.Sniffer().has_header(content)
            quotechar = dialect.quotechar
            doublequote = dialect.doublequote
            lineterminator = dialect.lineterminator
    except Exception as e:
        potential_delims = [',', ';', '\t', '|']
        delim_counts = {}
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                lines = [f.readline() for _ in range(10)]
                lines = [l for l in lines if l]
                for d in potential_delims:
                    counts = [l.count(d) for l in lines]
                    if any(c > 0 for c in counts):
                        delim_counts[d] = sum(counts) / len(counts)
            delimiter = max(delim_counts, key=delim_counts.get) if delim_counts else ','
        except:
            delimiter = ','
        has_header = True
        quotechar = '"'
        doublequote = True
        lineterminator = '\n'
        
    return {
        'encoding': encoding,
        'confidence': confidence,
        'delimiter': delimiter,
        'has_header': has_header,
        'quotechar': quotechar,
        'doublequote': doublequote,
        'lineterminator': repr(lineterminator),
        'size_mb': file_size_mb,
        'last_modified': last_mod,
        'abs_path': os.path.abspath(file_path)
    }


## 2. Structural Audit (Column Count Consistency)

This step checks if every row has the same number of columns as the header. It also detects if the file has multi-line records (embedded newlines in quotes).

In [99]:
def audit_structure(file_path: str, info: Dict[str, Any]):
    issues = []
    row_counts = []
    multi_line_rows = []
    
    with open(file_path, 'r', encoding=info['encoding']) as f:
        # Record raw line count vs CSV row count
        raw_lines = f.readlines()
        f.seek(0)
        
        reader = csv.reader(f, delimiter=info['delimiter'], quotechar=info['quotechar'])
        try:
            header = next(reader)
            expected_cols = len(header)
        except StopIteration:
            return "Empty File", []

        for i, row in enumerate(reader, 2):
            row_counts.append(len(row))
            if len(row) != expected_cols:
                issues.append({
                    'type': 'Column Mismatch', 
                    'row': i, 
                    'expected': expected_cols, 
                    'found': len(row), 
                    'content': row
                })
            
            # Check for multi-line records
            for cell in row:
                if '\n' in str(cell) or '\r' in str(cell):
                    multi_line_rows.append(i)
                    break

    summary = {
        'total_rows': len(row_counts),
        'expected_cols': expected_cols,
        'mismatched_rows': len(issues),
        'multi_line_rows_count': len(set(multi_line_rows)),
        'first_multiline_row': min(multi_line_rows) if multi_line_rows else None,
        'raw_line_count': len(raw_lines)
    }
    
    return summary, issues

path = os.path.join(test_dir, 'structural_issues.csv')
info = inspect_file_basics(path)
summary, issues = audit_structure(path, info)

print(f"Summary for {path}:")
for k, v in summary.items(): print(f"  {k}: {v}")

if issues:
    print("\nFirst 5 Structural Issues:")
    display(pd.DataFrame(issues).head(5))

Summary for test_files\structural_issues.csv:
  total_rows: 4
  expected_cols: 4
  mismatched_rows: 2
  multi_line_rows_count: 0
  first_multiline_row: None
  raw_line_count: 5

First 5 Structural Issues:


,type,row,expected,found,content
0,Column Mismatch,3,4,3,"[2, Bob, 25]"
1,Column Mismatch,4,4,5,"[3, Charlie, 35, Chicago, Extra]"


## 3. Data Integration & Type Inference

We load the file with Pandas. We'll attempt to infer the "true" type of each column and flag values that don't match (e.g., strings in a numeric column).

In [100]:
def get_quality_metrics(df: pd.DataFrame):
    metrics = []
    total_rows = len(df)
    
    for col in df.columns:
        nulls = df[col].isnull().sum()
        # Count empty strings or strings with only whitespace
        empties = 0
        if df[col].dtype == 'object':
            # Also check for 'nan' string which sometimes happens on bad loads
            empties = df[col].apply(lambda x: str(x).strip() == '' or str(x).lower() == 'nan').sum() - nulls
            empties = max(0, empties) # Ensure no negative if nulls overlap
            
        total_issues = nulls + empties
        
        metrics.append({
            'Column': col,
            'Nulls': nulls,
            'Null %': f"{(nulls/total_rows)*100:.2f}%" if total_rows > 0 else "0%",
            'Empty': empties,
            'Empty %': f"{(empties/total_rows)*100:.2f}%" if total_rows > 0 else "0%",
            'Total Quality Issues': total_issues,
            'Quality %': f"{(total_issues/total_rows)*100:.2f}%" if total_rows > 0 else "0%"
        })
    return pd.DataFrame(metrics)

def analyze_advanced_quality(df: pd.DataFrame):
    advanced_report = []
    
    # 1. Duplicates
    dup_count = df.duplicated().sum()
    
    for col in df.columns:
        # 2. Uniqueness
        unique_count = df[col].nunique()
        total_count = len(df)
        is_pk_candidate = (unique_count == total_count) and (df[col].isnull().sum() == 0)
        
        # 3. Whitespace / Hygiene
        whitespace_issues = 0
        if df[col].dtype == 'object':
            whitespace_issues = df[col].apply(lambda x: len(str(x)) != len(str(x).strip()) if pd.notnull(x) else False).sum()
        
        # 4. Outliers (Numeric only)
        outlier_count = 0
        numeric_vals = pd.to_numeric(df[col], errors='coerce').dropna()
        if not numeric_vals.empty:
            q1 = numeric_vals.quantile(0.25)
            q3 = numeric_vals.quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            outlier_count = ((numeric_vals < lower_bound) | (numeric_vals > upper_bound)).sum()
            
        advanced_report.append({
            'Column': col,
            'Unique Count': unique_count,
            'PK Candidate': is_pk_candidate,
            'Whitespace Issues': whitespace_issues,
            'Outlier Count': outlier_count
        })
        
    return dup_count, pd.DataFrame(advanced_report)

def analyze_column_types(df: pd.DataFrame):
    type_audit = []
    for col in df.columns:
        # Try to convert to numeric
        numeric_conv = pd.to_numeric(df[col], errors='coerce')
        num_valid = numeric_conv.notnull().sum()
        num_invalid = df[col].notnull().sum() - num_valid
        
        # Try to convert to datetime
        date_conv = pd.to_datetime(df[col], errors='coerce', format='mixed')
        date_valid = date_conv.notnull().sum()
        date_invalid = df[col].notnull().sum() - date_valid
        
        total_non_null = df[col].notnull().sum()
        
        detected_type = 'string'
        inconsistency_count = 0
        
        if num_valid > (total_non_null * 0.8) and total_non_null > 0:
            detected_type = 'numeric'
            inconsistency_count = num_invalid
        elif date_valid > (total_non_null * 0.8) and total_non_null > 0:
            detected_type = 'datetime'
            inconsistency_count = date_invalid
            
        type_audit.append({
            'Column': col,
            'Pandas Type': str(df[col].dtype),
            'Detected Type': detected_type,
            'Inconsistencies': inconsistency_count,
            'Nulls': df[col].isnull().sum()
        })
    
    return pd.DataFrame(type_audit)

path = os.path.join(test_dir, 'type_mismatch.csv')
df = pd.read_csv(path)
type_report = analyze_column_types(df)
print(f"\nType Audit for {path}:")
display(type_report)

def get_statistical_profile(df: pd.DataFrame, type_report: pd.DataFrame):
    """Generate mathematical summary for numeric columns detected via inference."""
    numeric_cols = type_report[type_report['Detected Type'] == 'numeric']['Column'].tolist()
    if not numeric_cols:
        return pd.DataFrame()
        
    stats = []
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if not series.empty:
            stats.append({
                'Column': col,
                'Min': series.min(),
                'Max': series.max(),
                'Mean': round(series.mean(), 2),
                'Std Dev': round(series.std(), 2),
                'Median': series.median()
            })
    return pd.DataFrame(stats)


def display_unique_values(df: pd.DataFrame, max_vals: int = 50):
    """Print a preview of unique values for each column."""
    print(f"\n[UNIQUE VALUES EXPLORATION] (Showing first {max_vals} unique values per column):")
    for col in df.columns:
        uniques = df[col].unique()
        count = len(uniques)
        preview = uniques[:max_vals]
        # Format for nicer printing
        preview_str = ", ".join([str(v) for v in preview])
        if count > max_vals:
            preview_str += " ..."
        print(f"  * {col} ({count} unique): {preview_str}")


def detect_redundancy(df: pd.DataFrame):
    """Identify columns that are 100% constant or 100% empty/null."""
    redundant = []
    for col in df.columns:
        null_count = df[col].isnull().sum()
        unique_vals = df[col].dropna().unique()
        
        if null_count == len(df):
            redundant.append({'Column': col, 'Issue': '100% Empty/Null'})
        elif len(unique_vals) == 1:
            redundant.append({'Column': col, 'Issue': f'Constant Value ({unique_vals[0]})'})
            
    return pd.DataFrame(redundant)

def validate_formats(df: pd.DataFrame):
    """Check for common format patterns like Email, Phone, URL using Regex."""
    patterns = {
        'Email': r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$',
        'URL': r'^https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+',
        'Phone (Simple)': r'^\+?1?\d{9,15}$'
    }
    
    format_issues = []
    # Only check string columns
    for col in df.select_dtypes(include=['object']).columns:
        for name, regex in patterns.items():
            # Sample non-null values
            sample = df[col].dropna().astype(str)
            if sample.empty: continue
            
            # If at least 10% of data matches, we assume this column SHOULD be this format
            matches = sample.str.match(regex).sum()
            if matches > (len(sample) * 0.1):
                mismatches = len(sample) - matches
                if mismatches > 0:
                    format_issues.append({
                        'Column': col,
                        'Expected Format': name,
                        'Invalid Count': mismatches,
                        'Invalid %': f"{(mismatches/len(df))*100:.2f}%"
                    })
    return pd.DataFrame(format_issues)

def track_history_metrics(file_path: str, row_count: int, null_count: int, size_mb: float):
    """Save current audit metrics to a history file and return alerts if trends are suspicious."""
    history_file = 'quality_history.json'
    alerts = []
    
    # Load history
    history = {}
    if os.path.exists(history_file):
        try:
            with open(history_file, 'r') as f:
                content = f.read()
                if content.strip():
                    history = json.loads(content)
        except (json.JSONDecodeError, ValueError):
            history = {} # Gracefully handle corrupt/empty files
            
    file_id = os.path.basename(file_path)
    prev = history.get(file_id)
    
    if prev:
        # Trend Analysis
        if row_count > prev['rows'] * 1.5 or row_count < prev['rows'] * 0.5:
            alerts.append(f"Significant Row Count Shift: {prev['rows']} -> {row_count}")
        if null_count > prev['nulls'] * 1.2:
            alerts.append(f"Null Spike Detected: {prev['nulls']} -> {row_count}")
            
    # Update history
    history[file_id] = {
        'rows': int(row_count),
        'nulls': int(null_count),
        'size': float(size_mb),
        'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    try:
        with open(history_file, 'w') as f:
            json.dump(history, f, indent=2)
    except Exception:
        pass # Don't let history save failure break the audit
        
    return alerts


def detect_placeholders(df: pd.DataFrame):
    """Search for common 'lazy' data entry placeholders."""
    placeholders = ['n/a', 'unknown', 'none', 'null', 'nan', '?', '-', '.', '999', '000', '0000-00-00']
    results = []
    for col in df.columns:
        # Check strings
        found_counts = {}
        sample = df[col].astype(str).str.lower().str.strip()
        for p in placeholders:
            count = (sample == p).sum()
            if count > 0:
                found_counts[p] = count
        
        if found_counts:
            results.append({
                'Column': col,
                'Placeholders Found': ", ".join([f"{k} ({v})" for k, v in found_counts.items()]),
                'Total Placeholder %': f"{(sum(found_counts.values())/len(df))*100:.2f}%"
            })
    return pd.DataFrame(results)

def analyze_headers(df: pd.DataFrame):
    """Audit column names for cleanliness and consistency."""
    header_issues = []
    for col in df.columns:
        issues = []
        if ' ' in col: issues.append("Contains Spaces")
        if any(not c.isalnum() and c not in ['_', '-'] for c in col): issues.append("Special Characters")
        if col.islower() and '_' not in col and len(col) > 10: issues.append("Potential missing snake_case")
        if col != col.strip(): issues.append("Leading/Trailing Whitespace")
        
        if issues:
            header_issues.append({'Column': col, 'Naming Issues': ", ".join(issues)})
    return pd.DataFrame(header_issues)

def get_categorical_distribution(df: pd.DataFrame, top_n: int = 3):
    """Get Top N most frequent values for categorical columns."""
    dist = []
    for col in df.columns:
        # Only check if unique count is relatively low compared to rows
        if df[col].nunique() < (len(df) * 0.5) or df[col].dtype == 'object':
            counts = df[col].value_counts(normalize=True).head(top_n)
            top_vals = [f"{val} ({perc:.1%})" for val, perc in counts.items()]
            dist.append({
                'Column': col,
                f'Top {top_n} Values': ", ".join(top_vals)
            })
    return pd.DataFrame(dist)




Type Audit for test_files\type_mismatch.csv:


,Column,Pandas Type,Detected Type,Inconsistencies,Nulls
0,id,int64,numeric,0,0
1,value,object,string,0,0
2,date,object,string,0,0


## 4. Comprehensive Audit & Report

Combining all checks into a final summary report.

In [101]:
def full_audit(file_path: str):
    print(f"\n{'='*60}")
    print(f"AUDIT REPORT: {os.path.basename(file_path)}")
    print(f"{'='*60}")
    
    # 1. Basics
    info = inspect_file_basics(file_path)
    print(f"[METADATA] Size: {info['size_mb']} MB | Modified: {info['last_modified']}")
    print(f"[METADATA] Path: {info['abs_path']}")
    print(f"[FILE] Encod: {info['encoding']} ({info['confidence']:.2%}) | Sep: '{info['delimiter']}'")
    
    # 2. Structure
    summary, issues = audit_structure(file_path, info)
    print(f"[STRUCTURE] Rows: {summary['total_rows']} | Cols: {summary['expected_cols']}")
    
    if summary['mismatched_rows'] > 0:
        mismatch_rows = [str(iss['row']) for iss in issues if iss['type'] == 'Column Mismatch']
        rows_str = ", ".join(mismatch_rows[:10]) + ("..." if len(mismatch_rows) > 10 else "")
        print(f"  !! WARNING: {summary['mismatched_rows']} rows have wrong column count! (Rows: {rows_str})")
    if summary['multi_line_rows_count'] > 0:
        print(f"  * INFO: Found {summary['multi_line_rows_count']} multi-line records.")
        if summary['first_multiline_row']:
            print(f"    -> First multiline record detected at row: {summary['first_multiline_row']}")
        
    # 3. Data Content
    try:
        df = pd.read_csv(file_path, encoding=info['encoding'], sep=info['delimiter'], on_bad_lines='skip')
        
        # Standard Quality Checks
        header_report = analyze_headers(df)
        empty_rows = df.isnull().all(axis=1).sum()
        placeholder_report = detect_placeholders(df)
        categorical_dist = get_categorical_distribution(df)
        
        # Advanced/Base Checks
        type_report = analyze_column_types(df)
        quality_report = get_quality_metrics(df)
        dup_total, adv_report = analyze_advanced_quality(df)
        history_alerts = track_history_metrics(file_path, len(df), type_report['Nulls'].sum(), info['size_mb'])
        redundant_report = detect_redundancy(df)
        format_report = validate_formats(df)
        
        if empty_rows > 0:
            print(f"  !! WARNING: {empty_rows} rows are ENTIRELY EMPTY (all columns Null)!")

        null_total = type_report['Nulls'].sum()
        inc_total = type_report['Inconsistencies'].sum()
        
        print(f"[CONTENT] Total Nulls: {null_total} | Type Inconsistencies: {inc_total}")
        
        stats_profile = get_statistical_profile(df, type_report)
        if not stats_profile.empty:
            print("\n[STATISTICAL PROFILE] Numeric Distribution Summary:")
            display(stats_profile)
            
        print("[QUALITY CHECK] Nulls & Empty Strings Details:")
        for _, row in quality_report.iterrows():
            print(f"[Column - {row['Column']}]  Nulls: {row['Nulls']} | %: {row['Null %']}")
            print(f"[Column - {row['Column']}]  Empty: {row['Empty']} | %: {row['Empty %']}")
            
        print(f"[ADVANCED] Duplicates: {dup_total} | PK Candidates: {adv_report['PK Candidate'].sum()}")
        
        if history_alerts:
            print("\n  !! TREND ALERTS:")
            for a in history_alerts: print(f"    - {a}")
            
        if not header_report.empty:
            print("\n[HEADER HYGIENE] Column Naming Warnings:")
            display(header_report)
            
        if not redundant_report.empty:
            print("\n[REDUNDANCY] Found Constant or Empty Columns:")
            display(redundant_report)
            
        if not placeholder_report.empty:
            print("\n[CONTENT HYGIENE] Potential Placeholders Found:")
            display(placeholder_report)
            
        if not categorical_dist.empty:
            print("\n[DATA DISTRIBUTION] Top Value Frequencies:")
            display(categorical_dist)
            
        if not format_report.empty:
            print("\n[FORMAT VALIDATION] Potential Regex Mismatches:")
            display(format_report)
            
        display_unique_values(df)
        
        # Display advanced highlights
        adv_issues = adv_report[(adv_report['Whitespace Issues'] > 0) | (adv_report['Outlier Count'] > 0)]
        if not adv_issues.empty:
            print("  -> Found hygiene or outlier issues:")
            display(adv_issues)
        
        if inc_total > 0:
            display(type_report[type_report['Inconsistencies'] > 0])
            
    except Exception as e:
        print(f"[ERROR] Could not load data with Pandas: {e}")

for f_name in sorted(os.listdir(test_dir)):
    if f_name.endswith('.csv'):
        full_audit(os.path.join(test_dir, f_name))


AUDIT REPORT: encoding_issue.csv
[METADATA] Size: 0.0001 MB | Modified: 2026-03-15 14:59:06
[METADATA] Path: d:\01.CodingProjects\DataQuality\test_files\encoding_issue.csv
[FILE] Encod: Windows-1252 (10.19%) | Sep: ','
[STRUCTURE] Rows: 3 | Cols: 3
[CONTENT] Total Nulls: 0 | Type Inconsistencies: 0

[STATISTICAL PROFILE] Numeric Distribution Summary:


,Column,Min,Max,Mean,Std Dev,Median
0,id,1,3,2.0,1.0,2.0


[QUALITY CHECK] Nulls & Empty Strings Details:
[Column - id]  Nulls: 0 | %: 0.00%
[Column - id]  Empty: 0 | %: 0.00%
[Column - name]  Nulls: 0 | %: 0.00%
[Column - name]  Empty: 0 | %: 0.00%
[Column - city]  Nulls: 0 | %: 0.00%
[Column - city]  Empty: 0 | %: 0.00%
[ADVANCED] Duplicates: 0 | PK Candidates: 3

[DATA DISTRIBUTION] Top Value Frequencies:


,Column,Top 3 Values
0,name,"Alice (33.3%), Bob (33.3%), Charlie (33.3%)"
1,city,"München (33.3%), São Paulo (33.3%), Tokyo (33.3%)"



[UNIQUE VALUES EXPLORATION] (Showing first 50 unique values per column):
  * id (3 unique): 1, 2, 3
  * name (3 unique): Alice, Bob, Charlie
  * city (3 unique): München, São Paulo, Tokyo

AUDIT REPORT: missing_values.csv
[METADATA] Size: 0.0001 MB | Modified: 2026-03-15 14:59:06
[METADATA] Path: d:\01.CodingProjects\DataQuality\test_files\missing_values.csv
[FILE] Encod: ascii (100.00%) | Sep: ','
[STRUCTURE] Rows: 4 | Cols: 4
[CONTENT] Total Nulls: 4 | Type Inconsistencies: 0

[STATISTICAL PROFILE] Numeric Distribution Summary:


...


[UNIQUE VALUES EXPLORATION] (Showing first 50 unique values per column):
  * id (3 unique): 1, 2, 4
  * name (3 unique): Alice, Bob, David
  * age (3 unique): 30, 25, 40
  * city (3 unique): New York, nan, Seattle

AUDIT REPORT: type_mismatch.csv
[METADATA] Size: 0.0001 MB | Modified: 2026-03-15 14:59:06
[METADATA] Path: d:\01.CodingProjects\DataQuality\test_files\type_mismatch.csv
[FILE] Encod: ascii (100.00%) | Sep: ','
[STRUCTURE] Rows: 4 | Cols: 3
[CONTENT] Total Nulls: 0 | Type Inconsistencies: 0

[STATISTICAL PROFILE] Numeric Distribution Summary:


,Column,Min,Max,Mean,Std Dev,Median
0,id,1,4,2.5,1.29,2.5


[QUALITY CHECK] Nulls & Empty Strings Details:
[Column - id]  Nulls: 0 | %: 0.00%
[Column - id]  Empty: 0 | %: 0.00%
[Column - value]  Nulls: 0 | %: 0.00%
[Column - value]  Empty: 0 | %: 0.00%
[Column - date]  Nulls: 0 | %: 0.00%
[Column - date]  Empty: 0 | %: 0.00%
[ADVANCED] Duplicates: 0 | PK Candidates: 3

[DATA DISTRIBUTION] Top Value Frequencies:


,Column,Top 3 Values
0,value,"100 (25.0%), abc (25.0%), 150.5 (25.0%)"
1,date,"2023-01-01 (25.0%), 2023-01-02 (25.0%), not-a-..."



[UNIQUE VALUES EXPLORATION] (Showing first 50 unique values per column):
  * id (4 unique): 1, 2, 3, 4
  * value (4 unique): 100, abc, 150.5, 200
  * date (4 unique): 2023-01-01, 2023-01-02, not-a-date, 2023-01-04
